# LSH Theory: (d₁,d₂,p₁,p₂)-Sensitive Families and Amplification

## Learning Objectives

By the end of this notebook you will be able to:
- State the formal definition of a locality-sensitive hash family
- Explain AND and OR constructions (amplification) and what each one does to error rates
- Derive the S-curve as the result of combined AND+OR amplification
- Choose band parameters (b, r) given target false-positive and false-negative rates
- Understand the information-theoretic lower bound on the number of hash functions required
- Identify the sensitive family that corresponds to MinHash, SimHash, and Hamming distance

## Definition: (d₁, d₂, p₁, p₂)-Sensitive Hash Family

A family **H** of functions h : 𝒳 → ℤ is **(d₁, d₂, p₁, p₂)-sensitive** for a distance measure D if for any two points x, y ∈ 𝒳:

| Condition | Guarantee |
|-----------|-----------|
| D(x, y) ≤ d₁ ("close") | P[h(x) = h(y)] ≥ p₁ |
| D(x, y) ≥ d₂ ("far")   | P[h(x) = h(y)] ≤ p₂ |

where h is chosen uniformly at random from H.

**Requirements for usefulness:**
- **p₁ > p₂** — close pairs must collide more often than far pairs
- **d₁ < d₂** — there must be a genuine gap between "close" and "far"

**Why the gap matters.** If p₁ and p₂ are close together (say 0.52 vs 0.48), the family barely distinguishes close from far points. We need them well-separated to achieve low error rates without an astronomical number of hash evaluations.

**Notation note.** The distance measure D need not be Euclidean — it can be Jaccard distance (1 − J), angular distance (θ/π), Hamming distance, or any other metric.

## MinHash as a Sensitive Family

Recall from earlier notebooks: for the MinHash function h defined by a random permutation π,

$$P[h(A) = h(B)] = \frac{|A \cap B|}{|A \cup B|} = J(A, B)$$

where J is the Jaccard similarity.

Translating to the framework above, use **Jaccard distance** D(A,B) = 1 − J(A,B):

- If D(A,B) ≤ d₁ (i.e. J ≥ 1 − d₁ = J₁), then P[h(A)=h(B)] = J(A,B) ≥ J₁ = p₁
- If D(A,B) ≥ d₂ (i.e. J ≤ 1 − d₂ = J₂), then P[h(x)=h(y)] = J(A,B) ≤ J₂ = p₂

So MinHash is a **(1−J₁, 1−J₂, J₁, J₂)-sensitive** family for Jaccard distance, or equivalently a **(J₁, J₂, J₁, J₂)-sensitive** family for Jaccard **similarity**.

**Limitation.** Since p₁ = J₁ and p₂ = J₂, the gap is exactly the gap between the two similarity thresholds you care about. For example, with J₁=0.7 and J₂=0.3, one hash function gives collision probabilities 0.7 vs 0.3 — a real gap, but not large enough to give good error rates on its own. That is where amplification comes in.

## Amplification via AND Construction

**Setup.** Draw r independent functions h₁, …, h_r from H. Form a compound function g(x) = (h₁(x), …, h_r(x)). Declare a collision for g iff ALL r component hashes agree.

**New collision probabilities:**

| Original condition | New probability |
|-------------------|-----------------|
| D(x,y) ≤ d₁ | P[g(x)=g(y)] ≥ p₁ʳ |
| D(x,y) ≥ d₂ | P[g(x)=g(y)] ≤ p₂ʳ |

This gives a new **(d₁, d₂, p₁ʳ, p₂ʳ)-sensitive** family H^AND.

**Effect.** Both probabilities decrease, but since p₁ > p₂ and both are in (0,1), we have:

$$\frac{p_1^r}{p_2^r} = \left(\frac{p_1}{p_2}\right)^r$$

The **ratio** p₁ʳ / p₂ʳ grows exponentially with r. The AND construction sharpens the distinction — at the cost of requiring more hash evaluations and driving both probabilities toward zero.

**Intuition.** Requiring all r functions to agree is like requiring r independent witnesses to all vouch for a pair. A true positive (close pair) has a good shot with each witness (p₁ each); a false positive (far pair) must fool every single witness (p₂ each).

## Amplification via OR Construction

**Setup.** Take b independent copies of H^AND (each copy being a band of r functions). Declare a collision if ANY of the b bands reports a collision.

**New collision probabilities:**

| Original condition | New probability |
|-------------------|-----------------|
| D(x,y) ≤ d₁ | P[at least one band matches] ≥ 1 − (1 − p₁ʳ)ᵇ |
| D(x,y) ≥ d₂ | P[at least one band matches] ≤ 1 − (1 − p₂ʳ)ᵇ |

**This is exactly the S-curve** from the bands-and-rows notebook! The combined AND+OR scheme (b bands of r hashes each, using k = b·r total hash functions) gives collision probability:

$$P[\text{candidate}] = 1 - \left(1 - J^r\right)^b$$

**Effect.** The OR construction drives both probabilities upward. Applied after AND, it rescues the true-positive rate from near-zero while keeping the false-positive rate manageable (because p₂ʳ is already tiny after AND).

**Summary:**
- AND alone → both probs shrink, ratio improves, but true positives are hurt
- OR alone → both probs grow, but close pairs benefit more
- AND then OR → creates the sharp sigmoid S-curve, concentrating decisions near the threshold

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt

def amplification_quality(b, r, p1, p2):
    """
    Returns (q1, q2) after b-OR of r-AND amplification.

    q1 = P[candidate | close pair]  (want this high)
    q2 = P[candidate | far pair]    (want this low)
    """
    # AND construction: p1^r, p2^r
    # OR  construction: 1-(1-p^r)^b
    q1 = 1 - (1 - p1**r)**b
    q2 = 1 - (1 - p2**r)**b
    return q1, q2

# MinHash with threshold t=0.5: p1=0.7 (close), p2=0.3 (far)
p1, p2 = 0.7, 0.3
k = 100  # total hash budget

print(f"Sweeping (b, r) with b*r = {k}, p1={p1}, p2={p2}")
print(f"{'b':>4} {'r':>4} {'q1':>8} {'q2':>8} {'q1-q2':>8} {'q1/q2':>8}")
print("-" * 50)

results = []
for r in range(1, k + 1):
    if k % r != 0:
        continue
    b = k // r
    q1, q2 = amplification_quality(b, r, p1, p2)
    results.append((b, r, q1, q2, q1 - q2, q1 / q2 if q2 > 1e-15 else float('inf')))

# sort by q1-q2 descending
results.sort(key=lambda x: -x[4])
for row in results[:15]:
    print(f"{row[0]:>4} {row[1]:>4} {row[2]:>8.4f} {row[3]:>8.4f} {row[4]:>8.4f} {row[5]:>8.1f}")

best = results[0]
print(f"\nBest by q1-q2: b={best[0]}, r={best[1]}")
print(f"  FPR = q2 = {best[3]:.4f}")
print(f"  FNR = 1-q1 = {1-best[2]:.4f}")

# Plot S-curves for a few (b,r) configurations
fig, ax = plt.subplots(figsize=(8, 5))
J_vals = np.linspace(0, 1, 200)
configs = [(b, r) for b, r, *_ in results[:5]]
# also add extreme cases
configs += [(1, k), (k, 1)]
for b, r in configs:
    curve = 1 - (1 - J_vals**r)**b
    ax.plot(J_vals, curve, label=f"b={b}, r={r}")
ax.axvline(p1, color='green', linestyle='--', alpha=0.6, label=f"p1={p1}")
ax.axvline(p2, color='red',   linestyle='--', alpha=0.6, label=f"p2={p2}")
ax.set_xlabel("Jaccard similarity J")
ax.set_ylabel("P[candidate pair]")
ax.set_title("S-curves for different (b,r) with b·r=100")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("lsh_theory_scurves.png", dpi=100)
plt.show()
print("S-curve plot saved.")


## False Positive and False Negative Rates

Given (b, r) and base probabilities (p₁, p₂):

$$\text{FPR} = P[\text{candidate} \mid D \geq d_2] = 1 - (1 - p_2^r)^b = q_2$$

$$\text{FNR} = P[\text{not candidate} \mid D \leq d_1] = 1 - q_1 = (1 - p_1^r)^b$$

**Design problem.** Given a requirement "FPR ≤ α and FNR ≤ β", find the minimum k = b·r.

Note the asymmetry in practice:
- False negatives are **permanent misses** — pairs you never re-examine.
- False positives are **cheap to fix** — you can verify every candidate with exact distance.
- Therefore: it usually makes more sense to allow higher FPR (α large) and demand low FNR (β small).


In [ ]:
def find_min_hashes(p1, p2, alpha, beta, k_max=1000):
    """
    Find (b, r) with minimum k=b*r such that FPR<=alpha and FNR<=beta.

    Returns list of (k, b, r, fpr, fnr) sorted by k.
    """
    solutions = []
    for k in range(1, k_max + 1):
        for r in range(1, k + 1):
            if k % r != 0:
                continue
            b = k // r
            q1, q2 = amplification_quality(b, r, p1, p2)
            fpr = q2
            fnr = 1 - q1
            if fpr <= alpha and fnr <= beta:
                solutions.append((k, b, r, fpr, fnr))
        if solutions:
            # found solutions at this k; return the best ones
            break
    return solutions

# Example: p1=0.7, p2=0.3, require FPR<=0.05 and FNR<=0.10
p1, p2 = 0.7, 0.3
alpha, beta = 0.05, 0.10
solutions = find_min_hashes(p1, p2, alpha, beta)

if solutions:
    print(f"Target: FPR≤{alpha}, FNR≤{beta}")
    print(f"  Minimum k = {solutions[0][0]}")
    print(f"  {'k':>4} {'b':>4} {'r':>4} {'FPR':>8} {'FNR':>8}")
    for sol in solutions:
        print(f"  {sol[0]:>4} {sol[1]:>4} {sol[2]:>4} {sol[3]:>8.4f} {sol[4]:>8.4f}")
else:
    print("No solution found in range.")

# Sweep different (alpha, beta) targets
print("\nMinimum k for various (alpha, beta) targets:")
print(f"  {'alpha':>6} {'beta':>6} {'min_k':>6} {'b':>4} {'r':>4}")
for alpha in [0.10, 0.05, 0.01]:
    for beta in [0.10, 0.05, 0.01]:
        sols = find_min_hashes(p1, p2, alpha, beta, k_max=500)
        if sols:
            k, b, r, fpr, fnr = sols[0]
            print(f"  {alpha:>6.2f} {beta:>6.2f} {k:>6} {b:>4} {r:>4}")
        else:
            print(f"  {alpha:>6.2f} {beta:>6.2f}   >500")


## Lower Bound: Why You Cannot Do Much Better

**Informal argument (Indyk-Motwani 1998).** Consider any LSH scheme based on k independent hash evaluations that achieves:
- FNR ≤ δ (miss at most δ fraction of close pairs)
- FPR ≤ ε (flag at most ε fraction of far pairs as candidates)

Then k must satisfy:

$$k = \Omega\!\left(\frac{\log(1/\delta)}{\log(p_1/p_2)}\right)$$

**Interpretation.**
- The denominator log(p₁/p₂) captures the "quality" of the base hash family — how well a single hash discriminates close from far. For MinHash at threshold 0.5, p₁/p₂ = 0.7/0.3 ≈ 2.33.
- The numerator log(1/δ) captures how reliably you need to find close pairs.
- You cannot beat this bound with any combination of AND/OR constructions using independent draws from H.

**Consequence.** The AND+OR (band) scheme is essentially optimal for independent hashes from a fixed family H. The only way to do better is to use a fundamentally better base family H with a larger p₁/p₂ ratio — which means using a different hash function suited to a different distance measure.

**Practical implication.** If the lower bound says you need k=100, then tuning (b,r) is just optimizing constants — you cannot escape needing roughly 100 hash evaluations to meet your error budget.

## Summary: Sensitive Families for Common Distance Measures

| Distance Measure | Hash Family | P[h(x)=h(y)] for similar pairs | P[h(x)=h(y)] for dissimilar pairs | Notes |
|---|---|---|---|---|
| **Jaccard** (sets) | MinHash (random permutation) | ≈ J₁ (high Jaccard) | ≈ J₂ (low Jaccard) | Exact: P = J(A,B) |
| **Cosine / Angular** (vectors) | SimHash (random hyperplane sign) | 1 − θ₁/π | 1 − θ₂/π | θ = angle between vectors |
| **Hamming** (binary strings) | Random bit selection: h_i(x) = x_i | 1 − d₁/L | 1 − d₂/L | L = string length, d = #differing bits |
| **Euclidean** (ℝᵈ) | p-stable projection + bucketing | Depends on w/‖u−v‖ | Decreases with distance | w = bucket width |

**Key takeaway:** the AND+OR amplification framework is universal — once you have any (d₁, d₂, p₁, p₂)-sensitive family H, you can tune (b, r) to hit any (FPR, FNR) target. The base family just needs p₁ > p₂ with a meaningful gap.

The next notebook (`ml_002_05_lsh_families.ipynb`) implements SimHash and p-stable projections in full.